# Part B Schritt fuer Schritt

Dieses Notebook zeigt die Preprocessing-Stufe aus `aibip.part_b` mit Zwischenergebnissen.

Ziel von Part B:
- Rohbild laden
- festen ROI-Ausschnitt waehlen: Himmel oben und Fahrzeughaube unten groesstenteils entfernen
- Ausschnitt auf Arbeitsbreite 2048 skalieren
- CLAHE anwenden, um Schatten und lokale Helligkeitsunterschiede abzumildern
- einfache QC/Analyse zeigen: Flags, FFT, Gaussian/Sobel/Canny

Hinweis: Die Beispielbilder werden aus `data/mini` gelesen; alternativ eigene Pfade in `EXAMPLE_PATHS` eintragen.

In [ ]:
from pathlib import Path
import sys

# Notebook kann aus notebooks/ gestartet werden; Repo-Root in sys.path aufnehmen.
ROOT = Path.cwd()
if not (ROOT / "aibip").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import cv2
import numpy as np
from IPython.display import display

# Wichtig im Notebook: Inline-Backend setzen, bevor pyplot importiert wird.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

import matplotlib.pyplot as plt

from aibip import config, imgio
from aibip.part_b import (
    crop_roi,
    resize_band,
    apply_clahe,
    preprocess,
    hood_strip_stats,
    flag_outliers,
    _road_band_crop,
    _log_spectrum,
)

plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["axes.titlesize"] = 10

print("Repo:", ROOT)
print("ROI:", config.ROI_TOP_FRAC, "bis", config.ROI_BOTTOM_FRAC)
print("WORK_WIDTH:", config.WORK_WIDTH, "PATCH:", config.PATCH)

## 1. Beispielbilder auswaehlen

Diese Liste enthaelt die Beispielbilder. Wenn einzelne Dateien fehlen, werden sie uebersprungen.

In [ ]:
EXAMPLE_PATHS = [
    Path("data/mini/WE1MY5SK.jpg"),
    Path("data/mini/WE1MY5SL.jpg"),
    Path("data/mini/WE1MY5SP.jpg"),
    Path("data/mini/WE1MY5SN.jpg"),
    Path("data/mini/WE1MY5SM.jpg"),
    Path("data/mini/WE1MY5SR.jpg"),
    Path("data/mini/WE1MY5SQ.jpg"),
    Path("data/mini/WE1MY5SO.jpg"),
    Path("data/mini/WE1MY5SS.jpg"),
    Path("data/mini/WE1MY5ST.jpg"),
]

paths = [p for p in EXAMPLE_PATHS if p.exists()]

if not paths:
    fallback = ROOT / "data" / "mini"
    paths = imgio.list_images(fallback)[:10] if fallback.exists() else []

if not paths:
    raise FileNotFoundError("Keine Beispielbilder gefunden. Lege Bilder in data/mini oder passe EXAMPLE_PATHS an.")

print(f"Gefundene Bilder: {len(paths)}")
for p in paths:
    print("-", p)

## 2. Rohbilder ansehen

OpenCV laedt Bilder als BGR. Fuer Matplotlib wandeln wir nach RGB um.

In [ ]:
def bgr_to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def show_images(items, cols=3, title=None):
    rows = int(np.ceil(len(items) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.5 * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (name, img) in zip(axes.ravel(), items):
        ax.imshow(bgr_to_rgb(img))
        ax.set_title(name)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    display(fig)
    plt.close(fig)

raw = {p.stem: imgio.load_image(p) for p in paths}
preview_raw = [(name, imgio.resize(img, (384, 512))) for name, img in list(raw.items())[:6]]
show_images(preview_raw, cols=3, title="Rohbilder, verkleinert fuer die Ansicht")

## 3. ROI-Crop sichtbar machen

`crop_roi()` nimmt den vertikalen Bereich von 40 Prozent bis 80 Prozent der Bildhoehe. Oben wird viel Himmel entfernt, unten wird die Haube weitgehend entfernt. Die roten Linien zeigen die Schnittkanten.

In [ ]:
def roi_overlay(img):
    view = imgio.resize(img, (768, 1024))
    h, w = view.shape[:2]
    y0 = int(h * config.ROI_TOP_FRAC)
    y1 = int(h * config.ROI_BOTTOM_FRAC)
    out = view.copy()
    out[:y0] = (out[:y0] * 0.35).astype(np.uint8)
    out[y1:] = (out[y1:] * 0.35).astype(np.uint8)
    cv2.line(out, (0, y0), (w - 1, y0), (0, 0, 255), 3)
    cv2.line(out, (0, y1), (w - 1, y1), (0, 0, 255), 3)
    cv2.putText(out, f"top {config.ROI_TOP_FRAC:.2f}", (12, max(24, y0 - 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
    cv2.putText(out, f"bottom {config.ROI_BOTTOM_FRAC:.2f}", (12, max(24, y1 - 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
    return out

show_images([(name, roi_overlay(img)) for name, img in list(raw.items())[:6]], cols=3, title="ROI-Crop: behalten wird der helle Mittelbereich")

## 4. Crop, Resize und CLAHE einzeln ausfuehren

Hier sieht man die drei Hauptschritte von `preprocess()` einzeln fuer ein Bild.

In [ ]:
name = list(raw.keys())[0]
img = raw[name]

cropped = crop_roi(img)
resized = resize_band(cropped)
clahe = apply_clahe(resized)
full_preprocess = preprocess(img)

print("raw:", img.shape)
print("cropped:", cropped.shape)
print("resized:", resized.shape)
print("clahe/preprocess:", clahe.shape, full_preprocess.shape)

show_images([
    ("1 raw", imgio.resize(img, (384, 512))),
    ("2 crop_roi", imgio.resize(cropped, (384, 512))),
    ("3 resize_band", imgio.resize(resized, (384, 512))),
    ("4 apply_clahe", imgio.resize(clahe, (384, 512))),
], cols=2, title=f"Part B Zwischenschritte fuer {name}")

## 5. CLAHE A/B Vergleich

Links ist der ROI-Ausschnitt ohne CLAHE, rechts mit CLAHE. Das ist wichtig, weil Schatten ein Confounder sind: Schatten koennen spaeter hohe Rekonstruktionsfehler erzeugen, obwohl die Strasse nicht kaputt ist.

In [ ]:
pairs = []
for name, img in list(raw.items())[:4]:
    band = resize_band(crop_roi(img))
    with_clahe = apply_clahe(band)
    combined = np.hstack([band, with_clahe])
    pairs.append((f"{name}: ohne CLAHE | mit CLAHE", imgio.resize(combined, (384, 1024))))

show_images(pairs, cols=1, title="CLAHE Vergleich")

## 6. Ergebnis von `preprocess()` fuer alle Beispielbilder

Das ist genau das Bild, das Part C spaeter fuer MobileSAM bekommt.

In [ ]:
processed = {name: preprocess(img) for name, img in raw.items()}
processed_preview = [(name, imgio.resize(img, (307, 1024))) for name, img in processed.items()]
show_images(processed_preview, cols=2, title="preprocess(image) -> image")

## 7. QC: Hood-Strip-Statistiken und Flags

Der Code misst im Rohbild einen Streifen knapp ueber der unteren ROI-Grenze. Dort sollte normalerweise Strasse liegen. Wenn dieser Streifen sehr hell/dunkel oder sehr gesaettigt ist, kann das auf Haubenreste oder andere Stoerungen hinweisen.

`flag_outliers()` nutzt robuste z-Scores ueber Median/MAD.

In [ ]:
records = []
for name, img in raw.items():
    stats = hood_strip_stats(img)
    records.append({"image_id": name, **stats})

for r in records:
    print(f"{r['image_id']}: V={r['v']:.1f}, S={r['s']:.1f}")

flags = flag_outliers(records)
print("\nFlags:")
if flags:
    for f in flags:
        print(f)
else:
    print("Keine robusten Ausreisser in dieser kleinen Stichprobe.")

## 8. Fourier-Analyse

Die Aufgabenstellung verlangt eine Frequenzanalyse. Hier vergleichen wir das Spektrum des ganzen vorbereiteten Bilds mit einem Road-Band-Crop. Asphalt sollte eher feinkoernige, relativ isotrope Textur zeigen; Vegetation und Kanten erzeugen staerkere Strukturen.

In [ ]:
pick = list(processed.items())[:3]
fig, axes = plt.subplots(len(pick), 3, figsize=(15, 4 * len(pick)), squeeze=False)

for i, (name, proc) in enumerate(pick):
    gray = cv2.cvtColor(proc, cv2.COLOR_BGR2GRAY).astype(np.float32)
    road_crop = _road_band_crop(proc)
    road_gray = cv2.cvtColor(road_crop, cv2.COLOR_BGR2GRAY).astype(np.float32)

    frame = proc.copy()
    h, w = frame.shape[:2]
    cv2.rectangle(frame, (int(w * 0.25), int(h * 0.60)), (int(w * 0.75), int(h * 0.95)), (0, 0, 255), 3)

    axes[i, 0].imshow(bgr_to_rgb(frame))
    axes[i, 0].set_title(f"{name}: Road-Band-Crop")
    axes[i, 1].imshow(_log_spectrum(gray), cmap="gray")
    axes[i, 1].set_title("FFT ganzes preprocess-Bild")
    axes[i, 2].imshow(_log_spectrum(road_gray), cmap="gray")
    axes[i, 2].set_title("FFT Road-Band")

for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
display(fig)
plt.close(fig)

## 9. Klassische Filter: Gaussian, Sobel, Canny

Das zeigt, welche Bildstrukturen klassische Filter hervorheben. Kanten entstehen stark an Bordsteinen, Markierungen, Fahrzeugen, Schatten und Vegetation. Genau deshalb wird spaeter die Strassenmaske aus Part C gebraucht.

In [ ]:
name = list(processed.keys())[0]
crop = _road_band_crop(processed[name])
gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

blurred = cv2.GaussianBlur(crop, (5, 5), 1.5)
sobel_x = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
sobel_mag = np.sqrt(sobel_x ** 2 + sobel_y ** 2)
sobel_mag = (sobel_mag / (sobel_mag.max() + 1e-9) * 255).astype(np.uint8)
edges = cv2.Canny(gray, 50, 150)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
panels = [
    ("Road crop", bgr_to_rgb(crop), None),
    ("Gaussian blur", bgr_to_rgb(blurred), None),
    ("Sobel magnitude", sobel_mag, "gray"),
    ("Canny edges", edges, "gray"),
]
for ax, (title, panel, cmap) in zip(axes, panels):
    ax.imshow(panel, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
display(fig)
plt.close(fig)

## 10. Ergebnisse speichern

Diese Zelle speichert eine kleine Montage der Part-B-Zwischenergebnisse in `outputs/notebook_part_b/`.

In [ ]:
out_dir = ROOT / "outputs" / "notebook_part_b"
out_dir.mkdir(parents=True, exist_ok=True)

tiles = []
for name, img in list(raw.items())[:6]:
    before = imgio.resize(img, (384, 512))
    after = imgio.resize(preprocess(img), (384, 512))
    tile = np.hstack([before, after])
    cv2.putText(tile, f"{name}: raw | preprocess", (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
    tiles.append(tile)

montage = imgio.montage(tiles, cols=1, pad=8)
out_path = out_dir / "raw_vs_preprocess.jpg"
imgio.save_image(out_path, montage)
print("Gespeichert:", out_path)
plt.figure(figsize=(12, 14))
plt.imshow(bgr_to_rgb(montage))
plt.axis("off")
display(plt.gcf())
plt.close()

## 11. Optional: die offiziellen Part-B-Kommandos aus dem Repo

Wenn `data/mini` vorhanden ist, erzeugen diese Kommandos dieselben Report-Artefakte wie im Projekt-Runbook:

```powershell
.\.venv\Scripts\python.exe -m aibip.part_b figures data\mini
.\.venv\Scripts\python.exe -m aibip.part_b preview data\mini --n 12
```

Unter macOS/Linux steht im Projekt noch `.venv/bin/python`; unter Windows ist meistens `.\.venv\Scripts\python.exe` richtig.